# Dag 3: MLflow logging + tracking

**Nøglebegreber:** `log_metric`, `log_param`, `log_model`, `autolog`, runs, experiments

**Læringsmål:**
- Forstå forholdet mellem MLflow experiments og runs
- Logge metrics, parametre og modeller manuelt med MLflow API
- Aktivere `mlflow.autolog()` og forstå hvad det automatisk fanger
- Hente og sammenligne runs via MLflow tracking client
- Opdatere `src/train.py` til at bruge `mlflow.start_run()` korrekt
- Submitte et command job og verificere MLflow-data i Azure ML Studio

**Forudsætninger:** Dag 1 (data assets), Dag 2 (environments + command jobs)

## 1. MLClient
Opret forbindelse til workspace

In [6]:
import sys
sys.path.append("..")

from src.utils import init_ml_client
ml_client = init_ml_client()

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


## 2. MLflow: Experiments og Runs

I MLflow er et **experiment** en samling af **runs**. Et run er en enkelt kørsel af din træningskode, og det indeholder:
- **Parametre** (`mlflow.log_param`) — hyperparametre, konfigurationsværdier
- **Metrics** (`mlflow.log_metric`) — talværdier der måler performance, f.eks. accuracy
- **Artefakter** (`mlflow.log_artifact`) — filer som plots, modeller, datasæt
- **Tags** (`mlflow.set_tag`) — fri tekst til at organisere runs

I Azure ML er hvert **job** automatisk et MLflow run, og jobbet's `experiment_name` svarer til MLflow experiment-navnet.

> **Eksamen tip:** `mlflow.log_metric(key, value, step=None)` — `step` er valgfrit og bruges til at logge metrics over tid (f.eks. per epoch). `mlflow.log_param` må kun kaldes **een gang per run** per nøgle — gentagne kald med samme nøgle kaster en fejl.

**Opgave:** Konfigurer MLflow tracking URI til dit Azure ML workspace og opret et eksperiment.

*Hint:* `ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri` giver dig tracking URI'en.

In [7]:
import mlflow

# Hent MLflow tracking URI fra Azure ML workspace
# TODO: Hent tracking URI via ml_client og sæt den med mlflow.set_tracking_uri()
tracking_uri = ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri
mlflow.set_tracking_uri(tracking_uri)

print("Tracking URI:", tracking_uri)

# TODO: Opret (eller aktiver) et experiment med mlflow.set_experiment()
# HINT: Brug experiment_name = "attrition-experiment"
experiment_name = "attrition-experiment"
mlflow.set_experiment(experiment_name)

print("Aktivt experiment:", experiment_name)

Tracking URI: azureml://swedencentral.api.azureml.ms/mlflow/v2.0/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/providers/Microsoft.MachineLearningServices/workspaces/data-scientist-cert
Aktivt experiment: attrition-experiment


## 3. Manuel logging: log_param og log_metric

Du kan logge direkte fra en notebook ved at starte et run med `mlflow.start_run()`. Dette opretter et nyt run under det aktive experiment.

> **Eksamen tip:** `mlflow.log_params(dict)` logger flere parametre på een gang. `mlflow.log_metrics(dict, step=None)` gør det samme for metrics. Begge er genveje til at undgå gentagne enkelt-kald.

**Opgave:** Start et MLflow run og log parametre, metrics og et tag manuelt.
- Log parametrene `reg=0.1` og `solver="liblinear"`
- Log metrics `val_accuracy=0.87` og `val_auc=0.76`
- Sæt et tag `model_type` til `"LogisticRegression"`

*Hint:* Brug `with mlflow.start_run() as run:` for automatisk at lukke runnet.

In [3]:
# Manuel logging af parametre og metrics
with mlflow.start_run(run_name="manual-logging-demo") as run:
    mlflow.log_params({"reg": 0.1, "solver": "liblinear"})
    mlflow.log_metrics({"val_accuracy": 0.87, "val_auc": 0.76})
    mlflow.set_tag("model_type", "LogisticRegression")

    run_id = run.info.run_id
    print("Run ID:", run_id)
    print("Run status:", run.info.status)

Run ID: 85e433d3-f5f4-4f2f-bc02-b5bbd4744b26
Run status: RUNNING
🏃 View run manual-logging-demo at: https://swedencentral.api.azureml.ms/mlflow/v2.0/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/providers/Microsoft.MachineLearningServices/workspaces/data-scientist-cert/#/experiments/bbb105d5-208c-4c8d-b8f0-e129ce829ad8/runs/85e433d3-f5f4-4f2f-bc02-b5bbd4744b26
🧪 View experiment at: https://swedencentral.api.azureml.ms/mlflow/v2.0/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/providers/Microsoft.MachineLearningServices/workspaces/data-scientist-cert/#/experiments/bbb105d5-208c-4c8d-b8f0-e129ce829ad8


## 4. log_artifact og log_model

`mlflow.log_artifact(local_path)` gemmer en fil (f.eks. et plot eller en CSV) som artefakt i runnet.

`mlflow.sklearn.log_model(model, artifact_path)` gemmer en scikit-learn model i MLflow's model format. Dette er forskelligt fra blot at gemme en pickle-fil — MLflow tilføjer metadata om environment og input-schema, som bruges ved deployment.

> **Eksamen tip:** `log_model` vs. `save_model`: `log_model` logger modellen som en artefakt i et aktivt run. `save_model` gemmer lokalt til disk uden at kræve et aktivt run. I Azure ML bruges `log_model` til at gøre modeller tilgængelige for model registry.

**Opgave:** Træn en simpel model lokalt og log den med `mlflow.sklearn.log_model()`.
1. Indlæs churn-data fra `../data/churn-raw.csv`
2. Træn en `LogisticRegression` model
3. Log modellen med `mlflow.sklearn.log_model(model, artifact_path="model")`
4. Log et feature importance-plot som artefakt

*Hint:* Brug `matplotlib` til at lave et simpelt bar-plot og gem det med `plt.savefig()` inden `mlflow.log_artifact()`.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import mlflow.sklearn
from pathlib import Path
from mlflow.types.schema import Schema, ColSpec

# used for model signature
input_schema = Schema([
    ColSpec("integer", "Age"),
    ColSpec("integer", "WorkLifeBalance"),
    ColSpec("integer", "YearsSinceLastPromotion"),
    ColSpec("integer", "JobInvolvement"),
    ColSpec("integer", "YearsAtCompany"),
    ColSpec("integer", "MonthlyIncome"),
    ColSpec("integer", "Gender_Female"),
    ColSpec("integer", "Department_Human Resources"),
    ColSpec("integer", "Department_Research & Development"),
    ColSpec("integer", "Department_Sales"),
])

# used for model signature
output_schema = Schema([ColSpec("boolean")])

# model signature
from mlflow.models.signature import ModelSignature
signature = ModelSignature(inputs=input_schema, outputs=output_schema)


def make_dummies(df: pd.DataFrame, categorical_columns: list[str]):
    for col in categorical_columns:
        temp = df[col]
        dummies = pd.get_dummies(temp, prefix=col)
        df = pd.concat([df, dummies], axis=1)

    df.drop(columns=categorical_columns, inplace=True)

    return df

with mlflow.start_run(run_name="log-model-demo-2") as run:
    csv_path = "../data/churn-raw.csv"
    df = pd.read_csv(csv_path)
    keep_cols = ['Attrition', 'Age','Gender','Department','WorkLifeBalance','YearsSinceLastPromotion','JobInvolvement','YearsAtCompany','MonthlyIncome']
    df_reduced = df[keep_cols]
    categorical_cols = ['Gender', 'Department']
    df_reduced = make_dummies(df_reduced, categorical_columns=categorical_cols)
    feature_names = df_reduced.drop(columns=['Attrition']).columns
    X, y = df_reduced[feature_names].values, df_reduced['Attrition'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)
    reg = 0.05
    C = 1.0 / reg
    model = LogisticRegression(C=C, solver="liblinear").fit(X_train, y_train)
    y_hat = model.predict(X_test)
    acc = float(np.average(y_hat == y_test))
    y_scores = model.predict_proba(X_test)[:, 1]
    auc = float(roc_auc_score(y_test, y_scores))

    mlflow.log_params({"reg": reg, "C": C})
    mlflow.log_metrics({"val_accuracy": acc, "val_auc": auc})
    mlflow.sklearn.log_model(model, artifact_path="model", signature=signature)

    # Lav et bar-plot af model.coef_[0] og gem det lokalt som "coef_plot.png"
    # Log derefter filen med mlflow.log_artifact()
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(feature_names, model.coef_[0])
    ax.set_title("Logistic Regression Coefficients")
    ax.set_xlabel("Feature")
    ax.set_ylabel("Coefficient")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plot_path = "../data/coef_plot.png"
    plt.savefig(plot_path)
    plt.close()

    mlflow.log_artifact(plot_path)

    model_run_id = run.info.run_id
    print("Run ID:", model_run_id)
    print(f"Accuracy: {acc:.4f}, AUC: {auc:.4f}")
    print("Model og plot logget som artefakter.")

2026/02/23 11:03:46 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Run ID: 81d53df7-27df-44ad-b686-7ace23b05c89
Accuracy: 0.8413, AUC: 0.7033
Model og plot logget som artefakter.
🏃 View run log-model-demo-2 at: https://swedencentral.api.azureml.ms/mlflow/v2.0/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/providers/Microsoft.MachineLearningServices/workspaces/data-scientist-cert/#/experiments/bbb105d5-208c-4c8d-b8f0-e129ce829ad8/runs/81d53df7-27df-44ad-b686-7ace23b05c89
🧪 View experiment at: https://swedencentral.api.azureml.ms/mlflow/v2.0/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/providers/Microsoft.MachineLearningServices/workspaces/data-scientist-cert/#/experiments/bbb105d5-208c-4c8d-b8f0-e129ce829ad8


## 5. mlflow.autolog()

`mlflow.autolog()` aktiverer automatisk logging for understøttede frameworks (scikit-learn, XGBoost, PyTorch, etc.). Det logger automatisk:
- Alle hyperparametre fra modellen
- Evalueringsmetrics (f.eks. training score)
- Modellen selv som artefakt
- Feature importance (for understøttede modeller)

> **Eksamen tip:** `mlflow.autolog()` skal kaldes **inden** modellen trænes. Du kan finjustere det med parametre som `log_models=True/False`, `log_input_examples=True/False`, og `disable=True` for at slå det fra igen. I Azure ML aktiveres autolog automatisk i command jobs medmindre du eksplicit deaktiverer det.

**Opgave:** Brug `mlflow.autolog()` til at logge en ny model uden manuelle kald.
- Aktiver autolog inden træning
- Træn en `RandomForestClassifier` med `n_estimators=100, max_depth=5`
- Bekræft at parametre og metrics er logget automatisk

*Hint:* Kig på `mlflow.sklearn.autolog()` for sklearn-specifik kontrol.

In [9]:
from sklearn.ensemble import RandomForestClassifier
mlflow.sklearn.autolog(log_models=True, log_input_examples=False)

with mlflow.start_run(run_name="autolog-demo") as run:
    rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    rf_model.fit(X_train, y_train)

    # Autolog har nu logget parametre og metrics automatisk.
    # Vi logger stadig vores egne test-metrics manuelt:
    y_hat_rf = rf_model.predict(X_test)
    acc_rf = float(np.average(y_hat_rf == y_test))
    y_scores_rf = rf_model.predict_proba(X_test)[:, 1]
    auc_rf = float(roc_auc_score(y_test, y_scores_rf))
    mlflow.log_metrics({"val_accuracy": acc_rf, "val_auc": auc_rf})

    autolog_run_id = run.info.run_id
    print("Run ID:", autolog_run_id)
    print(f"RF Accuracy: {acc_rf:.4f}, AUC: {auc_rf:.4f}")

# Deaktiver autolog igen for at undgå uventet adfærd i resten af notebooken
mlflow.sklearn.autolog(disable=True)

2026/02/23 11:05:40 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 0.24.1 <= scikit-learn <= 1.6.1, but the installed version is 1.8.0. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2026/02/23 11:05:49 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/02/23 11:05:52 WARNING mlflow.sklearn: Failed to log evaluation dataset information to MLflow Tracking. Reason: BAD_REQUEST: Response: {'Error': {'Code': 'UserError', 'Severity': None, 'Message': 'Cannot log the same dataset with different context', 'MessageFormat': None, 'MessageParameters': None, 'ReferenceCode': None, 'DetailsUri': None, 'Target': None, 'Details': [], 'InnerError': None, 'DebugInfo': None, 'AdditionalInfo': None}, 'Correlation': {'operation': 'ba8fe70e0b784f6d185cb6b7b66a

Run ID: 2a1b9004-b647-44c8-a5bb-3e040181ffc7
RF Accuracy: 0.8435, AUC: 0.6850
🏃 View run autolog-demo at: https://swedencentral.api.azureml.ms/mlflow/v2.0/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/providers/Microsoft.MachineLearningServices/workspaces/data-scientist-cert/#/experiments/bbb105d5-208c-4c8d-b8f0-e129ce829ad8/runs/2a1b9004-b647-44c8-a5bb-3e040181ffc7
🧪 View experiment at: https://swedencentral.api.azureml.ms/mlflow/v2.0/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/providers/Microsoft.MachineLearningServices/workspaces/data-scientist-cert/#/experiments/bbb105d5-208c-4c8d-b8f0-e129ce829ad8


## 6. Hent og sammenlign runs via MlflowClient

MLflow's `MlflowClient` giver programmatisk adgang til experiments og runs. Du kan søge efter runs, filtrere på metrics, og sammenligne resultater.

> **Eksamen tip:** `mlflow.search_runs(experiment_names=[...], filter_string="metrics.val_accuracy > 0.85")` returnerer en pandas DataFrame med alle matchende runs. `order_by=["metrics.val_auc DESC"]` sorterer efter en metric. Dette er nyttigt til at finde det bedste run automatisk.

**Opgave:** Brug `mlflow.search_runs()` til at finde alle runs i `attrition-experiment` og print de 3 bedste på `val_auc`.

*Hint:* `mlflow.search_runs(experiment_names=["attrition-experiment"], order_by=["metrics.val_auc DESC"])` returnerer en DataFrame.

In [30]:
runs_df = mlflow.search_runs(max_results=500)

# Filtrér på navn og status
names_to_show = ["autolog-demo", "log-model-demo-2"]
filtered = runs_df[(runs_df["status"] == "FINISHED") & (runs_df['tags.mlflow.runName'].isin(names_to_show))].copy()
filtered = filtered.sort_values("metrics.val_auc", ascending=False)
cols_to_show = ["tags.mlflow.runName", "run_id", "metrics.val_accuracy", "metrics.val_auc", "params.reg"]
available_cols = [c for c in cols_to_show if c in filtered.columns]
print(filtered[available_cols].to_string(index=False))

tags.mlflow.runName                               run_id  metrics.val_accuracy  metrics.val_auc params.reg
   log-model-demo-2 81d53df7-27df-44ad-b686-7ace23b05c89              0.841270         0.703273       0.05
       autolog-demo 2a1b9004-b647-44c8-a5bb-3e040181ffc7              0.843537         0.685002       None


## 7. Opdater train.py med mlflow.start_run()

Det nuværende `src/train.py` bruger `mlflow.log_metric()` og `mlflow.log_param()` — men det mangler:
1. Et eksplicit `mlflow.start_run()` kontekst-manager (Azure ML starter automatisk et run, men eksplicit brug giver mere kontrol)
2. `mlflow.log_artifact()` til at gemme et feature importance-plot
3. Et run-navn sat med `mlflow.set_tag("mlflow.runName", ...)`

> **Eksamen tip:** I et Azure ML command job behøver du ikke selv kalde `mlflow.set_tracking_uri()` — Azure ML sætter automatisk tracking URI via environment-variabler. Du skal blot importere mlflow og logge.

**Opgave:** Omskriv `src/train.py` så `main()` funktionen er pakket i `with mlflow.start_run():` og tilføj logging af et coefficients-plot som artefakt.

*Hint:* Sæt `run_name` i `mlflow.start_run(run_name=f"lr-reg-{args.reg}")` for at give runs informative navne.

In [50]:
%%writefile ../src/train.py
import argparse
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Ikke-interaktiv backend til server-miljø
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.sklearn
from mlflow.types.schema import Schema, ColSpec
from mlflow.models.signature import ModelSignature

# used for model signature
input_schema = Schema([
    ColSpec("integer", "Age"),
    ColSpec("integer", "WorkLifeBalance"),
    ColSpec("integer", "YearsSinceLastPromotion"),
    ColSpec("integer", "JobInvolvement"),
    ColSpec("integer", "YearsAtCompany"),
    ColSpec("integer", "MonthlyIncome"),
    ColSpec("integer", "Gender_Female"),
    ColSpec("integer", "Department_Human Resources"),
    ColSpec("integer", "Department_Research & Development"),
    ColSpec("integer", "Department_Sales"),
])

# used for model signature
output_schema = Schema([ColSpec("boolean")])

# model signature
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

def make_dummies(df: pd.DataFrame, categorical_columns: list) -> pd.DataFrame:
    for col in categorical_columns:
        dummies = pd.get_dummies(df[col], prefix=col)
        df = pd.concat([df, dummies], axis=1)
    df.drop(columns=categorical_columns, inplace=True)
    return df


def get_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    print(f"Analyzing {len(df)} rows of data")
    return df


def log_coef_plot(model: LogisticRegression, feature_names: list, output_dir: Path) -> None:
    """Lav et coefficients-plot og log det som MLflow-artefakt."""
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(feature_names, model.coef_[0])
    ax.set_title("Logistic Regression Coefficients")
    ax.set_xlabel("Feature")
    ax.set_ylabel("Coefficient")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plot_path = output_dir / "coef_plot.png"
    plt.savefig(str(plot_path))
    plt.close(fig)
    mlflow.log_artifact(str(plot_path))
    print(f"Coefficient plot logget: {plot_path}")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_data", dest="input_data", type=str, required=True)
    parser.add_argument("--reg", dest="reg", type=float, default=0.01,
                        help="Regularization rate (inverse used for C)")
    parser.add_argument("--model_dir", type=str, required=True,
                        help="Directory to save model (AML output)")
    return parser.parse_args()


def main(args: argparse.Namespace) -> None:
    df = get_data(args.input_data)

    keep_cols = [
        "Attrition", "Age", "Gender", "Department", "WorkLifeBalance",
        "YearsSinceLastPromotion", "JobInvolvement", "YearsAtCompany", "MonthlyIncome"
    ]
    df = df[keep_cols]
    df = make_dummies(df, ["Gender", "Department"])

    X = df.drop(columns=["Attrition"]).values
    y = df["Attrition"].values
    feature_names = df.drop(columns=["Attrition"]).columns.tolist()

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

    C = 1.0 / float(args.reg)

    with mlflow.start_run(run_name=f"lr-reg-{args.reg}"):
        print(f"Training LogisticRegression with reg={args.reg}, C={C}")
        model = LogisticRegression(C=C, solver="liblinear").fit(X_train, y_train)

        y_hat = model.predict(X_test)
        acc = float(np.average(y_hat == y_test))
        print(f"Accuracy: {acc}")

        mlflow.log_param("reg", args.reg)
        mlflow.log_param("C", C)
        mlflow.log_metric("val_accuracy", acc)

        if len(np.unique(y_test)) == 2:
            y_scores = model.predict_proba(X_test)[:, 1]
            auc = float(roc_auc_score(y_test, y_scores))
            print(f"AUC: {auc}")
            mlflow.log_metric("val_auc", auc)

        out_dir = Path(args.model_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        # Log modellen med mlflow.sklearn.log_model
        mlflow.sklearn.log_model(model, artifact_path="model", signature=signature)
        print(f"Model logget til artifact_path='model'")

        # Log coefficient plot som artefakt
        log_coef_plot(model, feature_names, out_dir)

if __name__ == "__main__":
    args = parse_args()
    main(args)

Overwriting ../src/train.py


## 8. Submit command job og verificer MLflow-logging

Nu submitter vi det opdaterede `train.py` som et Azure ML command job. Når jobbet kører, kan du se alle MLflow-loggede data direkte i Azure ML Studio under "Experiments".

> **Eksamen tip:** I Azure ML er `experiment_name` i `command()` det samme som MLflow experiment-navnet. Alle runs fra samme experiment samles og kan sammenlignes i Studio's "Metrics" view.

**Opgave:** Submit et command job med det opdaterede script og `reg=0.05`.

*Hint:* Kopier din command job-kode fra Dag 2, og tilføj `--reg 0.05` til command-strengen.

In [35]:
from azure.ai.ml import command, Input, Output
from azure.ai.ml.constants import AssetTypes

# TODO: Submit et command job der kører det opdaterede train.py med --reg 0.05
# HINT: Genbrug strukturen fra Dag 2 - husk at tilføje --reg ${{inputs.reg}} ELLER hardcode --reg 0.05

job = command(
    name="attrition-mlflow-day-3",
    description="Dag 3: MLflow logging med sttart_run, log_model og log_artifact",
    code="../src",
    environment="custom-environment@latest",
    experiment_name="attrition-experiment",
    display_name="dag3-mlflow-logging",
    compute="my-cluster",
    command="python train.py --input_data ${{inputs.data}} --model_dir ${{outputs.model}} --reg 0.05",
    inputs={
        "data": Input(type=AssetTypes.URI_FILE, path="azureml:ibm-churn-file:1")
    },
    outputs={
        "model": Output(type=AssetTypes.URI_FOLDER)
    }
)

returned_job = ml_client.jobs.create_or_update(job)
print("Job name:", returned_job.name)
print("Studio URL:", returned_job.studio_url)

pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


Job name: attrition-mlflow-day-3
Studio URL: https://ml.azure.com/runs/attrition-mlflow-day-3?wsid=/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourcegroups/data-scientist-cert-rg/workspaces/data-scientist-cert&tid=fd728459-8712-40a7-973d-772eb8a1976d


## 9. Hent run-data fra et submitted job

Når jobbet er fuldført, kan du hente dets MLflow run-data programmatisk. Azure ML mapper job-navne til MLflow run-ID'er, så du kan tilgå metrics direkte.

**Opgave:** Vent på at jobbet er færdigt og hent dernæst metrics fra run'et via `mlflow.search_runs()`.

*Hint:* Brug `ml_client.jobs.stream(returned_job.name)` for at vente på completion og se logs i realtid.

In [42]:
# Vent på at jobbet er færdigt (streamer logs til notebook)
# ADVARSEL: Dette kan tage 5-10 minutter første gang (environment build)
ml_client.jobs.stream(returned_job.name)

RunId: attrition-mlflow-day-3
Web View: https://ml.azure.com/runs/attrition-mlflow-day-3?wsid=/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourcegroups/data-scientist-cert-rg/workspaces/data-scientist-cert

Execution Summary
RunId: attrition-mlflow-day-3
Web View: https://ml.azure.com/runs/attrition-mlflow-day-3?wsid=/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourcegroups/data-scientist-cert-rg/workspaces/data-scientist-cert



In [44]:
runs_df = mlflow.search_runs(max_results=500)

# Filtrér på navn og status
names_to_show = ["autolog-demo", "log-model-demo-2", "dag3-mlflow-logging"]
filtered = runs_df[(runs_df["status"] == "FINISHED") & (runs_df['tags.mlflow.runName'].isin(names_to_show))].copy()
filtered = filtered.sort_values("metrics.val_auc", ascending=False)
cols_to_show = ["tags.mlflow.runName", "run_id", "metrics.val_accuracy", "metrics.val_auc", "params.reg"]
available_cols = [c for c in cols_to_show if c in filtered.columns]
print(filtered[available_cols].to_string(index=False))

tags.mlflow.runName                               run_id  metrics.val_accuracy  metrics.val_auc params.reg
   log-model-demo-2 81d53df7-27df-44ad-b686-7ace23b05c89              0.841270         0.703273       0.05
dag3-mlflow-logging               attrition-mlflow-day-3              0.841270         0.703273       0.05
       autolog-demo 2a1b9004-b647-44c8-a5bb-3e040181ffc7              0.843537         0.685002       None


## 10. (Bonus) Iterer over reguleringsrate og sammenlign runs

**Opgave:** Submit tre command jobs med forskellige `reg`-værdier (f.eks. `0.001`, `0.01`, `0.1`) og brug `mlflow.search_runs()` til at finde den bedste model baseret på `val_auc`.

Dette er forløberen til Sweep Jobs (Dag 5), hvor Azure ML automatiserer denne søgning.

> **Eksamen tip:** Det er god praksis at give hvert run et unikt navn (f.eks. via `run_name` i `start_run()` eller via `display_name` i `command()`), så du nemt kan identificere dem i Studio og via `search_runs()`.

In [46]:
# TODO: Submit 3 command jobs med reg = 0.001, 0.01, 0.1
# HINT: Brug en for-loop og varier reg-værdien i command-strengen og display_name

reg_values = [0.001, 0.01, 0.1]
submitted_jobs = []

for reg in reg_values:
    # TODO: Udfyld command job konfiguration for denne reg-værdi
    bonus_job = command(
        name=f"attrition-day-3-reg-{str(reg).replace('.', '-')}",
        code="../src",
        environment="custom-environment@latest",
        experiment_name="attrition-experiment",
        display_name=f"dag3-bonus-reg-{reg}",
        compute="my-cluster",
        command=f"python train.py --input_data ${{{{inputs.data}}}} --model_dir ${{{{outputs.model}}}} --reg {reg}",
        inputs={"data": Input(type=AssetTypes.URI_FILE, path="azureml:ibm-churn-file:1")},
        outputs={"model": Output(type=AssetTypes.URI_FOLDER)}
    )
    returned = ml_client.jobs.create_or_update(bonus_job)
    submitted_jobs.append(returned)
    print(f"Submitted job for reg={reg}: {returned.name}")

print(f"\n{len(submitted_jobs)} jobs submitted. Vent paa de er faerdige i Azure ML Studio.")
print("Kør naeste celle naar alle jobs viser status 'Completed'.")

pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


Submitted job for reg=0.001: attrition-day-3-reg-0-001


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


Submitted job for reg=0.01: attrition-day-3-reg-0-01


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


Submitted job for reg=0.1: attrition-day-3-reg-0-1

3 jobs submitted. Vent paa de er faerdige i Azure ML Studio.
Kør naeste celle naar alle jobs viser status 'Completed'.


In [48]:
runs_df = mlflow.search_runs(max_results=500)

In [49]:
runs_df = mlflow.search_runs(max_results=500)

# Filtrér på navn og status
filtered = runs_df[(runs_df["status"] == "FINISHED") & (runs_df['run_id'].str.contains('day-3-reg'))].copy()
filtered = filtered.sort_values("metrics.val_auc", ascending=False)
cols_to_show = ["tags.mlflow.runName", "run_id", "metrics.val_accuracy", "metrics.val_auc", "params.reg"]
available_cols = [c for c in cols_to_show if c in filtered.columns]
print(filtered[available_cols].to_string(index=False))

 tags.mlflow.runName                    run_id  metrics.val_accuracy  metrics.val_auc params.reg
  dag3-bonus-reg-0.1   attrition-day-3-reg-0-1               0.84127         0.703312        0.1
dag3-bonus-reg-0.001 attrition-day-3-reg-0-001               0.84127         0.703273      0.001
 dag3-bonus-reg-0.01  attrition-day-3-reg-0-01               0.84127         0.703273       0.01


In [51]:
mlflow.search_runs(filter_string="metrics.val_auc > 0.70 AND params.reg = '0.05'")

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.val_auc,metrics.val_accuracy,params.C,params.reg,tags.mlflow.runName,tags.mlflow.user,tags.mlflow.rootRunId,tags.mlflow.note.content
0,27779767-f64f-4d08-b581-800928cff546,bbb105d5-208c-4c8d-b8f0-e129ce829ad8,FAILED,,2026-02-23 09:27:57.896000+00:00,2026-02-23 09:28:02.128000+00:00,0.703273,0.84127,20.0,0.05,log-model-demo,Morten Gade,27779767-f64f-4d08-b581-800928cff546,None
1,81d53df7-27df-44ad-b686-7ace23b05c89,bbb105d5-208c-4c8d-b8f0-e129ce829ad8,FINISHED,,2026-02-23 10:03:41.487000+00:00,2026-02-23 10:03:53.168000+00:00,0.703273,0.84127,20.0,0.05,log-model-demo-2,Morten Gade,81d53df7-27df-44ad-b686-7ace23b05c89,None
2,attrition-mlflow-dag3,bbb105d5-208c-4c8d-b8f0-e129ce829ad8,FAILED,,2026-02-23 11:04:44.509000+00:00,2026-02-23 11:07:09.553000+00:00,0.703273,0.84127,20.0,0.05,dag3-mlflow-logging,Morten Gade,attrition-mlflow-dag3,"Dag 3: MLflow logging med sttart_run, log_mode..."
3,attrition-mlflow-day-3,bbb105d5-208c-4c8d-b8f0-e129ce829ad8,FINISHED,,2026-02-23 12:13:46.075000+00:00,2026-02-23 12:16:26.312000+00:00,0.703273,0.84127,20.0,0.05,dag3-mlflow-logging,Morten Gade,attrition-mlflow-day-3,"Dag 3: MLflow logging med sttart_run, log_mode..."


## Refleksion

Besvar disse sporgsmaal i din egne ord (i en ny markdown-celle nedenfor):

1. Hvad er forskellen paa `mlflow.log_model()` og `mlflow.save_model()`? Hvornaar bruger du hvad?
2. Hvad logger `mlflow.autolog()` automatisk for scikit-learn modeller, som du ellers ville skulle logge manuelt?
3. Hvornaar er det nyttigt at bruge `mlflow.start_run()` eksplicit i et Azure ML command job?
4. Hvad er forskellen paa et MLflow **experiment** og et MLflow **run** i Azure ML-konteksten?
5. Hvordan ville du bruge `mlflow.search_runs()` til at automatisk vælge den bedste model fra en serie af træningsjobs?

1: mlflow.log_model() kan kun bruges i et job, og skriver til en intern path, mens mlflow.save_model() også kan bruges udenfor et job og skriver til disk (selvom jeg ikke er helt med på, hvorfor log_model IKKE skriver til disk, den skriver vel til et eller andet sted?)

2: parametre, metrikker og model

3: det ved jeg ikke... jeg vil som udgangspunkt altid bruge "with mlflow.start_run()" i mine .py scripts

4: et eksperiment er en samling af runs. man kan således kategorisere sine runs, og fx liste metrikker for X runs i et givet eksperiment

5: jeg ville bruge eksperiment-kategoriseringen til at filtrere runs, og vælge en metrik at sortere på. derefter søge, filtrere og arrangere

## Noglpunkter (DP-100 eksamen)

- **`mlflow.log_param(key, value)`** — logger en enkelt parameter. Brug `log_params(dict)` for flere ad gangen. Samme noegle maa kun logges een gang per run.
- **`mlflow.log_metric(key, value, step=None)`** — logger en metric. `step` bruges til tidsseriedata (f.eks. loss per epoch). Samme noegle maa logges flere gange.
- **`mlflow.log_artifact(local_path)`** — logger en lokal fil (plot, CSV, etc.) som artefakt i det aktive run.
- **`mlflow.sklearn.log_model(model, artifact_path)`** — logger en scikit-learn model med MLflow's standard model format. Giver bedre deploymentintegration end `save_model()`.
- **`mlflow.autolog()`** — automatisk logging for understoettede frameworks. Skal kaldes foer traening. Kan deaktiveres med `disable=True`.
- **`mlflow.start_run(run_name=...)`** — i Azure ML command jobs startes runs automatisk, men eksplicit brug giver kontrol over run-navn og hierarki.
- **`mlflow.search_runs(experiment_names=[...], filter_string=..., order_by=[...])`** — returnerer pandas DataFrame med matchende runs. Brugbart til automatisk model selection.
- **Azure ML + MLflow**: `experiment_name` i `command()` = MLflow experiment-navn. Tracking URI saettes automatisk i jobs — ingen manuel `set_tracking_uri()` noedvendig.
- **Fase 2 eksamenssektionen** (Explore data & Experiments, 20-25%) — MLflow tracking er centralt her.